# **CO-OPS Sea Level Trend and Change Calculation**

Last updated on: 10/05/2026

## Overview
NOAA's Center for Operational Oceanographic Products and Services (CO-OPS) is the authoritative source for accurate, reliable, and timely data on tides, water levels, currents, and other coastal oceanographic and meteorological conditions. Through its network of long-term water level gauges, CO-OPS provides essential information, including [coastal inundation data and sea level trends](https://tidesandcurrents.noaa.gov/sea_level_info.html) through its network of long-term water level gauges. This data is critical for tracking relative sea level rise, analyzing the increasing frequency of high-tide flooding, and measuring the impacts of extreme storm surges. Please see the latest [Notice of Services Update Memo](https://repository.library.noaa.gov/view/noaa/74092) for an overview of the newest methodology changes.

This notebook is designed to quickly calculate a trend or change amount using CO-OPS's methodology. Users can choose their station of interest here: [Sea Level Trends Map](https://tidesandcurrents.noaa.gov/trends-and-extremes/main-product.html?).

For this example we will use [Sewells Point, VA (8638610)](https://tidesandcurrents.noaa.gov/trends-and-extremes/main-product.html?mapTab=station&tab=trends&station-id=8638610&plot=relative-sea-level-change).

## Learning Outcomes
In this notebook users will be able to:
  1.  Explore historical sea level data
  2.  Learn how to calculate a Sea Level Trend
  3.  Learn how CO-OPS's calculates the amount of change over the period of record (POR) and since the National Tidal Datum Epoch (NTDE).

## Data Source and Documentations
Input and output documentation can be found in the [CO-OPS Data API](https://api.tidesandcurrents.noaa.gov/api/prod/), [CO-OPS Derived Product API v0.1](https://api.tidesandcurrents.noaa.gov/dpapi/prod/) and the [CO-OPS Metadata API](https://api.tidesandcurrents.noaa.gov/mdapi/prod/).

Standard templates are available within the [CO-OPS API URL Builder](https://tidesandcurrents.noaa.gov/api-helper/url-generator.html), but this notebook is designed to give you more flexibility through building our own API template to generate queries from our Data API, MetaData API, and Derived Product API.

All data used for this notebook come from the CO-OPS API.

---
## Prerequisites and running the Notebook
This notebook uses Python programming and users may run it either in **Google Colab** or **Jupyter Notebook**.

* **Jupyter Notebook:** Recommended if you prefer to work offline and manage libraries and dependencies personally.
* **Google Colab:** The best option for cloud-based instant access, seamless cooperation, and if you have a good internet connection.

This notebook is designed so that people with limited familiarity with Python can run it.

### Using Google Colab

There are two ways to open this notebook in Colab.

**Method 1: Open directly from GitHub (Recommended)**

This is the easiest way to *open* the notebook.

1.  Open [Google Colab](https://colab.research.google.com/).
2.  In the menu bar, go to **File > Open Notebook**.
3.  In the popup window, click on the **GitHub** tab.
4.  Paste the repository URL into the search bar and press Enter:
    `https://github.com/NOAA-CO-OPS/Coastal_Hazards_Example_Notebooks`
5.  Click on `SL_Trend_and_Change_Calculation_Notebook/SL_Trend_Calculation_Notebook.ipynb` to open it.

**Method 2: Download and Upload (All Files)**

This method downloads the entire Coastal_Hazards_Example_Notebooks repository.

1.  Go to the [main GitHub repository](https://github.com/NOAA-CO-OPS/Coastal_Hazards_Example_Notebooks).
2.  Click the green **Code** button, then select **Download ZIP**.
3.  Extract the ZIP file on your computer. This folder contains the notebook (`.ipynb`).
4.  Open [Google Colab](https://colab.research.google.com/).
5.  Go to **File > Upload notebook...** and select the `SL_Trend_Calculation_Notebook.ipynb` file from the folder you just extracted.

### Using Jupyter Notebook

This method downloads the entire Coastal_Hazards_Example_Notebooks repository.

1.  Go to the [main GitHub repository](https://github.com/NOAA-CO-OPS/Coastal_Hazards_Example_Notebooks).
2.  Click the green **Code** button, then select **Download ZIP**.
3.  Extract the ZIP file on your computer.
4.  Start Jupyter Notebook on your local machine and navigate to the extracted folder (`.../SL_Trend_and_Change_Calculation_Notebook/`) to open `SL_Trend_and_Change_Calculation_Notebook.ipynb`.

---

### ▶️ Running the Notebook

Once you have the notebook open:

1.  [Explore different station/s](https://tidesandcurrents.noaa.gov/sltrends/sltrends.html) and decide which you want to analyze.
2.  In **Step 3** of the notebook, change the **station variable** and **unit of interest**.
3.  In the top menu, click **Runtime** and choose **Run All**.
4.  You will see a green check mark to the left of each cell as it is successfully executed.

Users should be prepared to commit up to one hour to familiarize themselves with the notebook, run code and examine results.

## Software
This tutorial uses the following Python packages:
- noaa-coops
- datetime
- requests
- time
- pandas
- numpy
- plotly
- seaborn
- cmocean
- statsmodels
- warnings

## **Step 1**
## Install and Load Packages
Lets check if you have these packages installed. If not, this cell will install them.

In [1]:
import subprocess, sys
packages = ["datetime","requests","time","pandas","numpy","matplotlib","seaborn","cmocean","statsmodels","warnings","plotly","noaa-coops"]
for package in packages:
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

Now we load our packages and assign them names to use throughout the notebook.

In [3]:
# @title
import datetime as dt
import requests
import time as time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import matplotlib as mpl
import seaborn as sns
import cmocean
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.dates as mdates
import plotly.graph_objects as go
import plotly.colors as pc
from plotly.subplots import make_subplots
from noaa_coops import Station
import warnings

#this is to suppress user warnings
warnings.filterwarnings("ignore")

## **Step 2**
## Here we choose the station of interest and measurement unit i.e. 'english' (feet) or 'metric' (meter)
Click [here](https://tidesandcurrents.noaa.gov/trends-and-extremes/main-product.html?mapTab=station&tab=trends) to see a map of stations to get Station IDs

### The stations with the longest periods of record (greater than 95 years) are:
|Station ID and Name  | Station ID and Name | Station ID and Name|
|:--------------------|:--------------------|:--------------------|
|1612340	Honolulu, HI	| 8574680	Baltimore, MD	| 8771450	Galveston, TX |
|1617760	Hilo, HI	| 8575512	Annapolis, MD	| 9410170	San Diego, CA |
|8410140	Eastport, ME	|	8594900	Washington, DC	| 9410230	La Jolla, CA|
|8418150	Portland, ME	|	8638610	Sewells Point, VA	| 9410660	Los Angeles, CA|
|8443970	Boston, MA	| 8665530	Charleston, SC	| 9414290 San Francisco, CA|
|8452660	Newport, RI	| 8720030	Fernandina Beach, FL | 9439040	Astoria, OR|
|8518750	The Battery, NY	|	8724580	Key West, FL| 9447130	Seattle, WA|
|8534720	Atlantic City, NJ	|	8727520	Cedar Key, FL | 9450460	Ketchikan, AK|
|8557380	Lewes, DE	| 8729840	Pensacola, FL| 9451600	Sitka, AK |

In [171]:
#input station ID below
station = '8638610'

#options are metric or english
units = 'metric'

### Results will be in Meters and Millimeters/Year if 'metric' is chosen
### Results will be in Feet and Inches/Decade if 'english' is chosen

In [172]:
# @title Station Metadata
print(f"Fetching data for Station ID: {station}...\n")

# Initialize the station object (this downloads the metadata)
station_meta = Station(station)

# General Metadata
station_name = station_meta.name
print(f"Station Name: {station_name}")
print(f"Coordinates (Lat, Lon): {station_meta.lat_lon}")

metadata_dict = station_meta.metadata
established_date = metadata_dict.get('details', {}).get('established', 'Unknown')
print(f"Established Date: {established_date}\n")


# Datum & Epoch Information
# Access the specific 'datums' dictionary within the metadata
datums_meta = metadata_dict.get('datums', {})

# Grab the epoch info, defaulting to '1983-2001' if missing or '0000-0000'
epoch_info = datums_meta.get('epoch', '0000-0000')
if epoch_info == '0000-0000' or not epoch_info:
    epoch_info = '1983-2001'

epic_start = int(epoch_info[0:4])
epic_end = int(epoch_info[5:])
epic_length = epic_end - epic_start
epic_mid = int(epic_start + epic_length / 2)

print(f"Epoch: {epic_start}-{epic_end}; Epoch Mid: {epic_mid}")


# Control vs. Subordinate Station
# .strip() removes any accidental spaces from the API response
ctrl_station = datums_meta.get('ctrlStation', '').strip()

if not ctrl_station:
    # If the control station string is empty, this is a Control Station
    print(f"Hierarchy: Station {station} is a Control Station.")
else:
    # If a control station ID exists, this is a Subordinate Station
    print(f"Hierarchy: Station {station} is a Subordinate Station.")
    print(f"Control Station ID: {ctrl_station}")

    # Extract the comparison dates from the DatumAnalysisPeriod list
    analysis_period_list = datums_meta.get('DatumAnalysisPeriod', ['Dates not available'])
    if analysis_period_list:
        analysis_period = analysis_period_list
        print(f"Dates of Comparison: {analysis_period}")


Fetching data for Station ID: 8638610...

Station Name: Sewells Point
Coordinates (Lat, Lon): {'lat': 36.94278, 'lon': -76.32861}
Established Date: 1927-07-01 00:00:00

Epoch: 1983-2001; Epoch Mid: 1992
Hierarchy: Station 8638610 is a Control Station.


## **Step 3**
## Pulling in our data from the CO-OPS API

### First, we can grab the published  relative sea level (RSL) trend from CO-OPS DPAPI. This trend is a combination of the rising ocean surface and the vertical land motion (VLM) at the site.

In [173]:
try:
    product = "observedSL"
    affil = "us"
    server = "https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/product/"

    myurl = (server + product + ".json?station=" + station + "&affil=" + affil + '&units=' + units)
    print(myurl)

    # Use requests to get the data
    urlResponse = requests.get(myurl)
    content = urlResponse.json()

    # Assign data to variables
    mydata = content['SeaLvlTrendsObserved']

    # Use json_normalize to flatten the nested dictionaries into columns
    my_trend = pd.json_normalize(mydata)

    # Access the flattened column names
    station_info = my_trend['stationId'].iloc[0] + ' ' + my_trend['metadata.stationName'].iloc[0] + ', ' + my_trend['metadata.state'].iloc[0]
    print(station_info)
    start_trend = pd.to_datetime(my_trend['seaLvlTrend.startDate'].iloc[0])
    end_trend = pd.to_datetime(my_trend['seaLvlTrend.endDate'].iloc[0])
    trend_value = my_trend['seaLvlTrend.trend'].iloc[0]
    por_change = my_trend['seaLvlChange.changePOR'].iloc[0]
    ntde_change = my_trend['seaLvlChange.changeNTDE'].iloc[0]
    if pd.isna(trend_value) or trend_value == 'None':
      print('This station does not meet the data requirements for the calculation of an NTDE trend.')
    # Display the dataframe
    display(my_trend)

except Exception as e:
    print(f'Missing data for this station. Error details: {e}')


https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/product/observedSL.json?station=8638610&affil=us&units=metric
8638610 Sewells Point, VA


,stationId,monthlyMeansUrl,seasonalCycleMonth,metadata.stationName,metadata.state,metadata.annotation,metadata.trendUnits,metadata.seasonalUnits,metadata.changeUnits,metadata.latitude,...,seaLvlTrend.startDate,seaLvlTrend.endDate,seaLvlTrend.seasonalAverage,seaLvlChange.changeNTDE,seaLvlChange.changePOR,seaLvlChange.changeNTDEoffset,seaLvlChange.ntdeStartMidpoint,seaLvlChange.ntdeEndMidpoint,seaLvlChange.porStartMidpoint,seaLvlChange.porEndMidpoint
0,8638610,https://api.tidesandcurrents.noaa.gov/dpapi/pr...,"[{'month': 1, 'seasonalVar': -0.094, 'seasonal...",Sewells Point,VA,None,mm/yr,meters,meters,36.942778,...,01/15/1992,12/15/2025,0.042,0.194,0.502,0.0,07/15/1992,07/15/2023,02/15/1930,07/15/2023


In [174]:
# @title ### Next we will pull and plot the monthly Mean Sea Level (MSL) data

product='observedSL'
datum = 'msl'

server = 'https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/product/'

myurl = server + product+'.json?station='+station+'&details=monthlymeans&datum='+datum+'&units='+units
print(myurl)

# Use "requests" to get the Monthly MSL Data
urlResponse = requests.get(myurl)
content=urlResponse.json()

# Assign data to variables.
mydata = content['data']
is_single_trend = True

station_name = content['stationName']

# Make it a Dataframe
mydf = pd.DataFrame(mydata)

last_year = int(mydf['year'].iloc[-1])

#have data end at the end of 2025
mydf = mydf[mydf['year']<2026]

# Define y-axis label dynamically
if units == 'metric':
    y_label = f"Meters relative to MSL Datum ({epoch_info})"
else:
    y_label = f"Feet relative to MSL Datum ({epoch_info})"

#Focusing on the MSL
df_msl=mydf.loc[:,['year','month','msl']]
#placeholder to represent each month
df_msl['day'] = '01'
df_msl['MSL'] = pd.to_numeric(df_msl['msl'],downcast='float')
df_msl['year'] = df_msl['year'].astype('int')
df_msl['month'] = df_msl['month'].astype('int')
df_msl['date'] = pd.to_datetime(dict(year=df_msl.year,month=df_msl.month,day=df_msl.day))

# Create interactive plot with Matplotlib's default line color
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_msl['date'],
    y=df_msl['MSL'],
    mode='lines',
    name='Monthly MSL',
    line=dict(color='#1f77b4'),
    connectgaps=False # Add this to prevent connecting gaps
))

fig.update_layout(
    title={
        'text': f"{station_info}<br>Monthly Mean Sea Level over time",
        'x': 0.5,  # Center the title
        'xanchor': 'center'
    },
    xaxis_title="Year",
    yaxis_title=y_label,
    template='plotly_white',
    height=750,
    width=1250
)

fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=True)
fig.update_traces(connectgaps=False)

fig.show()

https://api.tidesandcurrents.noaa.gov/dpapi/prod/webapi/product/observedSL.json?station=8638610&details=monthlymeans&datum=msl&units=metric


## **Step 4**

## **Calculating trends in relative water levels**

## Statistical analysis and plotting of long-term single and split trends

#### Walking through the process of calculating the trends as seen in [CO-OPS Trends and Extremes Product](https://tidesandcurrents.noaa.gov/trends-and-extremes/main-product.html?mapTab=station&tab=trends)

#### *** The trend calculations in this notebook may vary from the trend calculations on the website listed above.


*   The primary difference is that this notebook is in Python and the trends on the website are calculated using the R programming language. Python and R have different packages/libraries with different capabilities. It is expected to see some variations in using different languages/packages for running the Autoregressive Integrated Moving Average (ARIMA) model. The ARIMA model call and order in this notebook is the same as that run in R to produce the trends on the website, just with different packages/libraries.

This section walks through how the sea level (SL) trends are calculated using an Autoregressive Integrated Moving Average (ARIMA) model.

Because we are focused strictly on the autoregressive component of the data, we utilize the simplest form of this model: ARIMA (1,0,0).

For a comprehensive explanation of our statistical methodology—including why we rely solely on the autoregressive (AR) term and how we account for seasonality using exogenous variables—please refer to our official reports:

*   [NOAA technical service publication NOS CO-OPS 004: "NOAA's sea level trends"](https://repository.library.noaa.gov/view/noaa/74092)
*   [NOAA Technical Report NOS CO-OPS 53: "Sea Level Variations of the United States 1854-2006"](https://tidesandcurrents.noaa.gov/publications/Tech_rpt_53.pdf)
*   [NOAA Technical Report NOS CO-OPS 36: "Sea Level Variations of the United States 1854-1999"](https://tidesandcurrents.noaa.gov/publications/NOAA_Technical_Report_NOS_COOPS_036.pdf)



Click [here](https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html#) for documentation on the ARIMA model in python.

In [175]:
trend_value = my_trend['seaLvlTrend.trend'].iloc[0]
if pd.isna(trend_value) or trend_value == 'None':
    print('This station does not meet the data requirements for the calculation of an NTDE trend.')
else:
  df_msl = df_msl.set_index(df_msl.month)

  #set index to date
  df_msl.set_index(df_msl.date,inplace=True)

  # We use a monthly frequency ('MS' for month-start).
  start_year = 1992
  end_date = df_msl.index.max()
  full_date_range = pd.date_range(start=f'{start_year}-01-01', end=end_date, freq='MS')

  # Reindex the MSL data to the full date range.
  # This creates a complete time series from 1992.
  # Any months without data in the original df_msl will have their 'MSL' value as NaN.
  msl_full_range = df_msl['MSL'].reindex(full_date_range)

  # Create a new DataFrame for the complete trend timeframe
  trend_timeframe = pd.DataFrame(msl_full_range)
  trend_timeframe.index.name = 'date'

  # Calculate the decimal date for the time variable in the model
  # (e.g., mid-February 2020 becomes 2020.125)
  trend_timeframe.index = pd.to_datetime({
          'year': trend_timeframe.index.year,
          'month': trend_timeframe.index.month,
          'day': 15
      })

  # The updated formula to account for the middle of the month
  trend_timeframe['decimal_date'] = trend_timeframe.index.year + (trend_timeframe.index.month - 1 + 0.5) / 12
  #print(trend_timeframe)

  #create matrix for ARIMA model, this is the format that the MSL data is in prior to the ARIMA model
  matrix_data = pd.DataFrame()
  matrix_data['time1']=trend_timeframe['decimal_date']-2000
  matrix_data['month']=trend_timeframe.index.month
  matrix_data['msl']=trend_timeframe['MSL']
  matrix_data['mon1']=matrix_data['month'].apply(lambda x: 1 if x == 1 else 0)
  matrix_data['mon2']=matrix_data['month'].apply(lambda x: 1 if x == 2 else 0)
  matrix_data['mon3']=matrix_data['month'].apply(lambda x: 1 if x == 3 else 0)
  matrix_data['mon4']=matrix_data['month'].apply(lambda x: 1 if x == 4 else 0)
  matrix_data['mon5']=matrix_data['month'].apply(lambda x: 1 if x == 5 else 0)
  matrix_data['mon6']=matrix_data['month'].apply(lambda x: 1 if x == 6 else 0)
  matrix_data['mon7']=matrix_data['month'].apply(lambda x: 1 if x == 7 else 0)
  matrix_data['mon8']=matrix_data['month'].apply(lambda x: 1 if x == 8 else 0)
  matrix_data['mon9']=matrix_data['month'].apply(lambda x: 1 if x == 9 else 0)
  matrix_data['mon10']=matrix_data['month'].apply(lambda x: 1 if x == 10 else 0)
  matrix_data['mon11']=matrix_data['month'].apply(lambda x: 1 if x == 11 else 0)
  matrix_data['mon12']=matrix_data['month'].apply(lambda x: 1 if x == 12 else 0)

  matrix_data = matrix_data.drop('month',axis=1)

  msl = pd.DataFrame(matrix_data['msl'])

  matrix_data = matrix_data.drop('msl',axis=1)

  exog=matrix_data

  if units == 'metric':
      model=ARIMA(endog=msl*1000,exog=exog,order=(1,0,0),trend='n')
  else:
  #if english, we want in inches instead of ft
    model=ARIMA(endog=msl*12,exog=exog,order=(1,0,0),trend='n')
  model_fit = model.fit()

  print(model_fit.summary())



                               SARIMAX Results                                
Dep. Variable:                    msl   No. Observations:                  408
Model:                 ARIMA(1, 0, 0)   Log Likelihood               -2292.617
Date:                Mon, 05 Oct 2026   AIC                           4615.234
Time:                        19:59:07   BIC                           4675.403
Sample:                             0   HQIC                          4639.043
                                - 408                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
time1          6.3245      0.525     12.047      0.000       5.296       7.353
mon1         -51.1935     12.312     -4.158      0.000     -75.325     -27.062
mon2         -40.8964     10.906     -3.750      0.0

In [176]:
# @title Plotting the trend and 95% confidence intervals
if pd.isna(trend_value) or trend_value == 'None':
  print('This station does not meet the data requirements for the calculation of an NTDE trend.')
else:
  seasonal_coefs = [model_fit.params[f'mon{i}'] for i in range(1, 13)]
  seasonal_avg = sum(seasonal_coefs) / 12
  months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

  if units == 'metric':
        # Convert from mm to meters
        zero_mean_seasonality = [round((coef - seasonal_avg) / 1000,3) for coef in seasonal_coefs]
        yaxis_title = 'Deviation from Seasonal Average (m)'
        yaxis_range = [-0.15, 0.15]
  else: # english
        # Convert from inches to feet
        zero_mean_seasonality = [round((coef - seasonal_avg) / 12,2) for coef in seasonal_coefs]
        yaxis_title = 'Deviation from Seasonal Average (ft)'
        yaxis_range = [-0.5, 0.5]
  #get info from model
  coefs=model_fit.params
  se=model_fit.bse

  #extracting our trend information from model
  if units == 'metric':
    trend_text=str(np.round(coefs[0],2)) +' +/- '+str(np.round(se[0]*1.96,2))+' mm/yr'
    slope_m = coefs[0]/1000
    error_m = (se[0]*1.96)/1000
  else:
    trend_text = f"{np.round(coefs[0] * 10, 2)} +/- {np.round(se[0] * 1.96 * 10, 2)} in/decade"
    slope_m = (coefs[0] / 12)
    error_m = (se[0] * 1.96 / 12)

  print(trend_text)

  #function to find y-int
  def getYInt(x1, y1,slope):
        y_intercept = slope * (-x1)+y1
        return (y_intercept)

  msl_adjust = df_msl[df_msl['date']>='1992-01-01']
  msl_adjust['decimal_date'] = msl_adjust.index.year + (msl_adjust.index.month - 1 + 0.5) / 12
  msl_adjust.index = pd.to_datetime({
        'year': msl_adjust.index.year,
        'month': msl_adjust.index.month,
        'day': 15
    })

  # Create a mapping from month (1–12) to seasonal mean
  month_to_seasonal = dict(zip(range(1, 13), zero_mean_seasonality))

  # Create the adjusted MSL column by subtracting the monthly seasonal mean
  msl_adjust['msl_seasonal_adjust'] = msl_adjust['MSL'].subtract(
        msl_adjust.index.month.map(month_to_seasonal)
    )

  #grabbing the months result from arima model
  seasonal = coefs[2:14]
  if units == 'metric':
    #calculating the mean seasonal and adding it to the trend line
    seasonal = np.mean(seasonal)/1000
  else:
    seasonal = np.mean(seasonal) / 12
  lin_y_int = getYInt(2000,seasonal,slope_m)

  #calculating linear trend
  msl_adjust['linear_trend'] = slope_m*msl_adjust['decimal_date']+lin_y_int
  #manually calculate se of estimate
  msl_adjust['y_y2']=(msl_adjust['msl_seasonal_adjust']-msl_adjust['linear_trend'])**2
  y_y2_sum = msl_adjust['y_y2'].sum()
  #print(y_y2_sum)
  msl_adjust['se_regression'] = np.sqrt(y_y2_sum / (len(msl_adjust) - 2))

  #calculate standard error of the regression line at each point
  one_over_n = 1/len(msl_adjust)
  msl_adjust['x_x2'] = (msl_adjust['decimal_date']-msl_adjust['decimal_date'].mean())**2
  msl_adjust['sum_x_x2'] = msl_adjust['x_x2'].sum()
  msl_adjust['sqrt'] = np.sqrt(one_over_n+(msl_adjust['x_x2']/msl_adjust['sum_x_x2']))
  msl_adjust['se_point'] = (msl_adjust['se_regression'])*msl_adjust['sqrt']

  # get variance inflation factor (VIF)
  p1 = coefs[-2]
  VIF = ((1 + p1) / (1 - p1)) ** 0.5
  # Calculate standard error of the regression line at each point
  msl_adjust['se_point_vif'] = msl_adjust['se_point'] * VIF
  #confidence bands
  msl_adjust['upr'] = msl_adjust['linear_trend']+1.96*msl_adjust['se_point_vif']
  msl_adjust['lwr'] = msl_adjust['linear_trend']-1.96*msl_adjust['se_point_vif']

6.32 +/- 1.03 mm/yr


In [177]:
# Updated code to not count NANs in data for CI interval
# Identify valid data to calculate proper N
valid_mask = msl_adjust['msl_seasonal_adjust'].notna()
n_valid = valid_mask.sum()

# calculating linear trend (applies to all points)
msl_adjust['linear_trend'] = slope_m * msl_adjust['decimal_date'] + lin_y_int

# manually calculate se of estimate
msl_adjust['y_y2'] = (msl_adjust['msl_seasonal_adjust'] - msl_adjust['linear_trend'])**2
# Only sum the residuals of valid data
y_y2_sum = msl_adjust.loc[valid_mask, 'y_y2'].sum()
#print(y_y2_sum)

# Using n_valid instead of len()
msl_adjust['se_regression'] = np.sqrt(y_y2_sum / (n_valid - 2))

# calculate standard error of the regression line at each point
one_over_n = 1 / n_valid

# Calculate the mean date ONLY using valid data points (Matches SQL t_bar)
mean_valid_date = msl_adjust.loc[valid_mask, 'decimal_date'].mean()

# Calculate the X deviations for ALL rows (so we can draw continuous bands)
msl_adjust['x_x2'] = (msl_adjust['decimal_date'] - mean_valid_date)**2

# Only sum the X deviations of the valid data points (Matches SQL t_sigma_squared)
sum_x_x2 = msl_adjust.loc[valid_mask, 'x_x2'].sum()
msl_adjust['sum_x_x2'] = sum_x_x2

# Apply the curve calculation to all rows using the valid denominators
msl_adjust['sqrt'] = np.sqrt(one_over_n + (msl_adjust['x_x2'] / msl_adjust['sum_x_x2']))
msl_adjust['se_point'] = msl_adjust['se_regression'] * msl_adjust['sqrt']

# get variance inflation factor (VIF)
p1 = coefs[-2]
VIF = ((1 + p1) / (1 - p1)) ** 0.5

# Calculate standard error of the regression line at each point
msl_adjust['se_point_vif'] = msl_adjust['se_point'] * VIF

# confidence bands
msl_adjust['upr'] = msl_adjust['linear_trend'] + 1.96 * msl_adjust['se_point_vif']
msl_adjust['lwr'] = msl_adjust['linear_trend'] - 1.96 * msl_adjust['se_point_vif']

In [178]:
# @title
if pd.isna(trend_value) or trend_value == 'None':
  print('This station does not meet the data requirements for the calculation of an NTDE trend.')
else:
  # Create figure
  fig = go.Figure()

  # Add traces
  fig.add_trace(go.Scatter(x=msl_adjust.index, y=msl_adjust["msl_seasonal_adjust"],
                            mode='lines', name='Monthly mean sea level with the average seasonal cycle removed',
                            line=dict(color='cornflowerblue')))

  fig.add_trace(go.Scatter(x=msl_adjust.index, y=msl_adjust["linear_trend"],
                            mode='lines', name='Linear Relative Sea Level Trend',
                            line=dict(color='red')))

  fig.add_trace(go.Scatter(x=msl_adjust.index, y=msl_adjust["upr"],
                            mode='lines', name='Upper 95% Confidence Interval',
                            line=dict(color='black')))

  fig.add_trace(go.Scatter(x=msl_adjust.index, y=msl_adjust["lwr"],
                            mode='lines', name='Lower 95% Confidence Interval',
                            line=dict(color='black')))

  # Y-axis label and limits
  if units == 'metric':
        yaxis_title = "Meters"
        if station in ['8771450', '8761724']:
            y_range = [-0.75, 0.6]
        elif station == '9453220':
            y_range = [-0.45, 1.35]
        elif station == '9455090':
            y_range = [-1.35, 0.45]
        else:
            y_range = [-0.6, 0.6]
  else:
        yaxis_title = "Feet"
        y_range = [-2.5, 2] if station in ['8771450','8761724'] else [-2, 2]

  # Update layout with grid lines
  fig.update_layout(
        title=dict(
            text=f"Relative Sea Level Trend<br><br>{station_info} &nbsp;&nbsp;&nbsp;&nbsp; {trend_text}",
            x=0.5,  # Center the title
            xanchor='center',
            font=dict(size=15)
        ),
        xaxis_title="Year",
        yaxis_title=yaxis_title,
        xaxis=dict(showgrid=True),
        yaxis=dict(showgrid=True, range=y_range, zeroline=True, zerolinecolor='black'),
        template="simple_white",
        legend=dict(x=0.01, y=0.99),
        width=1250,
        height=650
    )

  # Show the figure
  fig.show()

## **Step 5**

How has the **amount** of sea elvel changed over time?

Next we will calculate the amount of change for the station's period of record (POR) as well as since the NTDE.

In [179]:
# @title Change over the POR

# configurations
UNIT = "m"
station_info = station + " " + station_name
LINE_WIDTH_MONTHLY = 1
MONTHLY_OPACITY = 0.32
LINE_WIDTH_ANNUAL = 2
ANNUAL_OPACITY = 1.0
LINE_WIDTH_MARKER = 6
FONT_SIZE_TITLE = 32
FONT_SIZE_AXIS_TITLE = 22
FONT_SIZE_TICKS = 18
FONT_SIZE_ANNOTATION = 16
FONT_SIZE_LEGEND = 18
PLOT_WIDTH = 1250
PLOT_HEIGHT = 650

# Data processing & QC
# Ensure index is datetime
df_msl.index = pd.to_datetime(df_msl['date'])

# SLIDING WINDOW: Find the first 5-year period with 70% data (42 months)
required_months = 42
valid_start_date = None
valid_end_date = None
first_5_data = None

# Create a list of dates that actually have 'MSL' data in them
dates_with_data = df_msl['MSL'].dropna().index.sort_values().unique()

# Loop chronologically only through dates that have data
for current_date in dates_with_data:
    # Define the 5-year window for the current date
    end_date = current_date + pd.DateOffset(years=5)

    # Slice the data for this window and drop missing values
    window_data = df_msl[(df_msl.index >= current_date) & (df_msl.index < end_date)]['MSL'].dropna()

    # Check if this window meets the 70% criteria
    if len(window_data) >= required_months:
        valid_start_date = current_date
        valid_end_date = end_date
        first_5_data = window_data
        break # Exit the loop immediately since we found our first valid window!

# Handle the case where NO 5-year window meets the criteria
if valid_start_date is None:
    print(f"Data Availability Error: Could not find any 5-year period with at least {required_months} valid months.")
else:
    # Calculate the midpoint of the valid first 5 years
    mid_start_date = valid_start_date + (valid_end_date - valid_start_date) / 2

    # Calculate the end dates and end midpoint early so we can check our rules
    annual_avg_series = df_msl['MSL'].resample('YE').mean().dropna()
    end_dates = [annual_avg_series.index[-5], annual_avg_series.index[-1]]
    mid_end_date = end_dates[0] + (end_dates[1] - end_dates[0]) / 2

    # Calculate time differences for our validation checks
    years_diff = round((mid_end_date - mid_start_date).days / 365.25)

    absolute_data_start = df_msl.index.min()
    years_past_start = (mid_start_date - absolute_data_start).days / 365.25

    # Data validation check
    is_data_valid = True # We assume the data is good until proven otherwise

    # Check: Is the timeframe between midpoints less than 25 years?
    if years_diff < 25:
        print(f"Data Validation Error: Station fails criteria. The period between midpoints is only {years_diff} years (minimum 25)")
        is_data_valid = False

    # Only proceed with printing success and plotting if the data passed the check
    if is_data_valid:
        # Snap the printed dates strictly to the 15th of their respective months
        display_start_date = valid_start_date.replace(day=15)
        display_mid_date = mid_start_date.replace(day=15)

        print(f"The first valid 5-year period starts on {display_start_date.strftime('%Y-%m-%d')}.")
        print(f"--> The midpoint of this 5-year period is: {display_mid_date.strftime('%Y-%m-%d')}")

        # Calculate the two baseline averages for the plot
        avg_first_5 = first_5_data.mean()
        avg_last_5 = annual_avg_series.tail(5).mean()
        delta_por = avg_last_5 - avg_first_5

        start_dates = [valid_start_date, valid_end_date]

        # 5. Build the Plotly Figure
        fig = go.Figure()

        # Monthly Data
        fig.add_trace(go.Scatter(
            x=df_msl.index, y=df_msl['MSL'],
            mode='lines', name='Monthly Average',
            line=dict(color='royalblue', width=LINE_WIDTH_MONTHLY),
            opacity=MONTHLY_OPACITY
        ))

        # Horizontal Baseline (Connects start to the modern vertical)
        fig.add_trace(go.Scatter(
            x=[start_dates[0], mid_end_date],
            y=[avg_first_5, avg_first_5],
            mode='lines',
            name='Historical Baseline Level',
            line=dict(color='black', width=2, dash='dash'),
            showlegend=False, hoverinfo='skip'
        ))

        # The Vertical Line (The Red Measurement Line)
        fig.add_trace(go.Scatter(
            x=[mid_end_date, mid_end_date],
            y=[avg_first_5, avg_last_5],
            mode='lines',
            name='Amount of Change',
            line=dict(color='red', width=4),
            hoverinfo='skip'
        ))

        # Annual Trend
        fig.add_trace(go.Scatter(
            x=annual_avg_series.index, y=annual_avg_series,
            mode='lines', name='Annual Average',
            opacity=ANNUAL_OPACITY,
            line=dict(color='royalblue', width=LINE_WIDTH_ANNUAL)
        ))

        # 5-year Period Markers (The "Goalposts")
        fig.add_trace(go.Scatter(
            x=start_dates, y=[avg_first_5, avg_first_5],
            mode='lines', name='5-yr Average',
            line=dict(color='black', width=LINE_WIDTH_MARKER)
        ))

        fig.add_trace(go.Scatter(
            x=end_dates, y=[avg_last_5, avg_last_5],
            mode='lines', showlegend=False, name='5-yr Average',
            line=dict(color='black', width=LINE_WIDTH_MARKER)
        ))

        # Floating annotation
        fig.add_annotation(
            x=mid_end_date,
            y=(avg_first_5 + avg_last_5) / 2,
            text=f"<b>{delta_por:.2f}{UNIT} change over {years_diff} years</b><br><span style='font-size:12px'></span>",
            font=dict(size=FONT_SIZE_ANNOTATION, color="red"),
            showarrow=True,
            arrowhead=2,
            arrowcolor="red",
            ax=80,
            ay=0,
            xanchor="left",
            bgcolor="white",
            bordercolor="red",
            borderwidth=2,
            borderpad=10
        )

        fig.update_layout(
            title=dict(text=f"<b>Sea Level Change: {station_info}</b>", font=dict(size=FONT_SIZE_TITLE), x=0.5, y=0.95),
            xaxis=dict(
                title="<b>Year</b>",
                titlefont=dict(size=FONT_SIZE_AXIS_TITLE),
                tickfont=dict(size=FONT_SIZE_TICKS),
                showgrid=True,
                gridcolor='lightgray',
                range=[df_msl.index.min(), df_msl.index.max() + pd.Timedelta(days=365*8)]
            ),
            yaxis=dict(
                title=f"<b>Mean Sea Level ({UNIT})</b>",
                titlefont=dict(size=FONT_SIZE_AXIS_TITLE),
                tickfont=dict(size=FONT_SIZE_TICKS),
                showgrid=True,
                gridcolor='lightgray',
                range=[-0.6, 0.6]
            ),
            template="simple_white",
            width=PLOT_WIDTH,
            height=PLOT_HEIGHT,
            margin=dict(t=120, b=100, l=120, r=250),
            legend=dict(font=dict(size=FONT_SIZE_LEGEND), x=0.01, y=0.98, bordercolor="black", borderwidth=1, bgcolor="rgba(255,255,255,0.9)")
        )

        fig.show()

The first valid 5-year period starts on 1927-08-15.
--> The midpoint of this 5-year period is: 1930-01-15


For the change since NTDE calculation, we must first check if the station is on the current NTDE datum. For stations not on the NTDE, we must calculate an offset to bring the data relative to the NTDE.

In [180]:

# @title Change since NTDE

#printing info on control/subordinate status again
# Control vs. Subordinate Station
ctrl_station = datums_meta.get('ctrlStation', '').strip()
if not ctrl_station:
    print(f"Hierarchy: Station {station} is a Control Station.\n")
else:
    print(f"Hierarchy: Station {station} is a Subordinate Station.")
    print(f"Control Station ID: {ctrl_station}")
    analysis_period_list = datums_meta.get('DatumAnalysisPeriod', ['Dates not available'])
    if analysis_period_list:
        print(f"Dates of Comparison: {analysis_period_list}\n")

#first we need to check if our station requires an offset
if station in ['8770613','8771013','9455760']:
  print('This station is a subordinate station on a MTDE datum. We cannot accurately adjust the data to the NTDE.')
  avg_last_5 = np.nan
elif epic_mid != 1992 or station in ['8771450','8771510','8770570','8771341','8770822','8772471','9455920','9462620']:
  print('Station is not on the NTDE. Computing an offset based on the monthly means frim 1983-2001 to put station relative to the NTDE.')
  # Baseline calculation (1983-2001)
  epoch_data = df_msl.loc['1983-01-01':'2001-12-31']
  avg_msl_83_01 = epoch_data['MSL'].mean()

  # Apply Offset
  df_msl['MSL_Offset'] = df_msl['MSL'] - avg_msl_83_01

  msl_complete = df_msl

  # Annual Average Series
  annual_avg_series = msl_complete['MSL_Offset'].resample('YE').mean().dropna()

  # Calculate the two baseline averages
  avg_first_5 = first_5_data.mean()
  avg_last_5 = annual_avg_series.tail(5).mean()
  delta_por = avg_last_5 - avg_first_5

  # Define date ranges for the black markers
  # First 5-YEAR exact window check
  first_date = df_msl.index.min()
  end_first_5_date = first_date + pd.DateOffset(years=5)

  start_dates = [first_date, end_first_5_date]
  end_dates = [annual_avg_series.index[-5], annual_avg_series.index[-1]]

  # Calculate the midpoint of the final 5 years to drop the vertical crosshair line
  mid_end_date = end_dates[0] + (end_dates[1] - end_dates[0]) / 2

  # Define our dates for the baseline and the total change measurement
  epoch_start = "1983-01-01"
  epoch_end = "2001-12-31"
  plot_end_date = "2031-01-01"
  midpoint_23 = pd.Timestamp("2023-07-02")
  baseline_epoch = 0

  fig = go.Figure()

  # Monthly Data
  fig.add_trace(go.Scatter(
      x=msl_complete['1982':].index,
      y=msl_complete['1982':]['MSL_Offset'],
      mode='lines', name='Monthly Average',
      line=dict(color='royalblue', width=LINE_WIDTH_MONTHLY),
      opacity=MONTHLY_OPACITY
  ))

  # Annual Trend
  fig.add_trace(go.Scatter(
      x=annual_avg_series.index, y=annual_avg_series,
      mode='lines', name='Annual Average',
      opacity=ANNUAL_OPACITY,
      line=dict(color='royalblue', width=LINE_WIDTH_ANNUAL)
  ))

  # Highlight the 1983-2001 epoch (Solid Black Line)
  fig.add_trace(go.Scatter(
      x=[epoch_start, epoch_end],
      y=[baseline_epoch, baseline_epoch],
      mode='lines',
      name='1983-2001 Epoch',
      line=dict(color='black', width=LINE_WIDTH_MARKER),
      hoverinfo='skip'
  ))

  # Post-epoch baseline (Dashed Black Line)
  fig.add_trace(go.Scatter(
      x=[epoch_end, 2024],
      y=[baseline_epoch, baseline_epoch],
      mode='lines',
      name='Baseline Continuation',
      line=dict(color='black', width=3, dash='dash'),
      showlegend=False,
      hoverinfo='skip'
  ))

  # The vertical rise (Solid Red Measurement Line)
  fig.add_trace(go.Scatter(
      x=[midpoint_23, midpoint_23],
      y=[baseline_epoch, avg_last_5],
      mode='lines',
      name='Total Change',
      line=dict(color='red', width=4),
      hoverinfo='skip'
  ))

  # Modern Average Line (The heavy black bar)
  fig.add_trace(go.Scatter(
      x=[pd.Timestamp("2021-01-01"), pd.Timestamp("2025-12-31")],
      y=[avg_last_5, avg_last_5],
      mode='lines', name='Current 5yr Avg',
      line=dict(color='black', width=LINE_WIDTH_MARKER)
  ))

  # red annotation box
  fig.add_annotation(
      x=midpoint_23,
      y=(baseline_epoch + avg_last_5) / 2,
      text=f"<b>TOTAL CHANGE</b><br>{avg_last_5:.2f}m",
      font=dict(size=FONT_SIZE_ANNOTATION, color="red"),
      showarrow=True,
      arrowhead=2,
      arrowcolor="red",
      ax=120,
      ay=0,
      bgcolor="white",
      bordercolor="red",
      borderwidth=2,
      borderpad=10
  )

  # Layout
  fig.update_layout(
      title=dict(text=f"<b>Sea Level Change since NTDE (1983-2001)<br>{station_info}</b>", font=dict(size=FONT_SIZE_TITLE), x=0.5, y=0.95),
      xaxis=dict(
          title="<b>Year</b>",
          titlefont=dict(size=FONT_SIZE_AXIS_TITLE),
          tickfont=dict(size=FONT_SIZE_TICKS),
          range=["1982-01-01", plot_end_date],
          showgrid=True,
          gridcolor='lightgray'
      ),
      yaxis=dict(
          title="<b>Mean Sea Level (m)</b>",
          titlefont=dict(size=FONT_SIZE_AXIS_TITLE),
          tickfont=dict(size=FONT_SIZE_TICKS),
          showgrid=True,
          gridcolor='lightgray',
          zeroline=False
      ),
      template="simple_white",
      width=PLOT_WIDTH,
      height=PLOT_HEIGHT,
      margin=dict(t=120, b=100, l=120, r=250),
      legend=dict(
          font=dict(size=FONT_SIZE_LEGEND),
          x=0.01, y=0.98,
          bordercolor="black",
          borderwidth=1,
          bgcolor="rgba(255,255,255,0.9)"
      )
  )
  fig.show()
else:
  print('Station is on the current NTDE (1983-2001)')
  msl_complete = df_msl

  # Annual Average Series
  annual_avg_series = msl_complete["msl"].resample('YE').mean().dropna()

  # Calculate the two baseline averages
  avg_first_5 = first_5_data.mean()
  avg_last_5 = annual_avg_series.tail(5).mean()

  delta_por = avg_last_5 - avg_first_5

  # Define date ranges for the black markers
  # First 5-YEAR exact window check
  first_date = df_msl.index.min()
  end_first_5_date = first_date + pd.DateOffset(years=5)

  start_dates = [first_date, end_first_5_date]
  end_dates = [annual_avg_series.index[-5], annual_avg_series.index[-1]]

  # Calculate the midpoint of the final 5 years to drop the vertical crosshair line
  mid_end_date = end_dates[0] + (end_dates[1] - end_dates[0]) / 2

  # Define our dates for the baseline and the total change measurement
  epoch_start = "1983-01-01"
  epoch_end = "2001-12-31"
  plot_end_date = "2031-01-01"
  midpoint_23 = pd.Timestamp("2023-07-02")
  baseline_epoch = 0

  fig = go.Figure()

  # Monthly Data
  fig.add_trace(go.Scatter(
      x=df_msl['1982':].index,
      y=df_msl['1982':]['MSL'],
      mode='lines', name='Monthly Average',
      line=dict(color='royalblue', width=LINE_WIDTH_MONTHLY),
      opacity=MONTHLY_OPACITY
  ))

  # Annual Trend
  fig.add_trace(go.Scatter(
      x=annual_avg_series.index, y=annual_avg_series,
      mode='lines', name='Annual Average',
      opacity=ANNUAL_OPACITY,
      line=dict(color='royalblue', width=LINE_WIDTH_ANNUAL)
  ))

  # Highlight the 1983-2001 epoch (Solid Black Line)
  fig.add_trace(go.Scatter(
      x=[epoch_start, epoch_end],
      y=[baseline_epoch, baseline_epoch],
      mode='lines',
      name='1983-2001 Epoch',
      line=dict(color='black', width=LINE_WIDTH_MARKER),
      hoverinfo='skip'
  ))

  # Post-epoch baseline (Dashed Black Line)
  fig.add_trace(go.Scatter(
      x=[epoch_end, 2024],
      y=[baseline_epoch, baseline_epoch],
      mode='lines',
      name='Baseline Continuation',
      line=dict(color='black', width=3, dash='dash'),
      showlegend=False,
      hoverinfo='skip'
  ))

  # The vertical rise (Solid Red Measurement Line)
  fig.add_trace(go.Scatter(
      x=[midpoint_23, midpoint_23],
      y=[baseline_epoch, avg_last_5],
      mode='lines',
      name='Total Change',
      line=dict(color='red', width=4),
      hoverinfo='skip'
  ))

  # Modern Average Line (The heavy black bar)
  fig.add_trace(go.Scatter(
      x=[pd.Timestamp("2021-01-01"), pd.Timestamp("2025-12-31")],
      y=[avg_last_5, avg_last_5],
      mode='lines', name='Current 5yr Avg',
      line=dict(color='black', width=LINE_WIDTH_MARKER)
  ))

  # annotation box
  fig.add_annotation(
      x=midpoint_23,
      y=(baseline_epoch + avg_last_5) / 2,
      text=f"<b>TOTAL CHANGE</b><br>{avg_last_5:.2f}m",
      font=dict(size=FONT_SIZE_ANNOTATION, color="red"),
      showarrow=True,
      arrowhead=2,
      arrowcolor="red",
      ax=120,
      ay=0,
      bgcolor="white",
      bordercolor="red",
      borderwidth=2,
      borderpad=10
  )

  # layout
  fig.update_layout(
      title=dict(text=f"<b>Sea Level Change since NTDE (1983-2001)<br>{station_info}</b>", font=dict(size=FONT_SIZE_TITLE), x=0.5, y=0.95),
      xaxis=dict(
          title="<b>Year</b>",
          titlefont=dict(size=FONT_SIZE_AXIS_TITLE),
          tickfont=dict(size=FONT_SIZE_TICKS),
          range=["1982-01-01", plot_end_date],
          showgrid=True,
          gridcolor='lightgray'
      ),
      yaxis=dict(
          title="<b>Mean Sea Level (m)</b>",
          titlefont=dict(size=FONT_SIZE_AXIS_TITLE),
          tickfont=dict(size=FONT_SIZE_TICKS),
          showgrid=True,
          gridcolor='lightgray',
          zeroline=False
      ),
      template="simple_white",
      width=PLOT_WIDTH,
      height=PLOT_HEIGHT,
      margin=dict(t=120, b=100, l=120, r=250),
      legend=dict(
          font=dict(size=FONT_SIZE_LEGEND),
          x=0.01, y=0.98,
          bordercolor="black",
          borderwidth=1,
          bgcolor="rgba(255,255,255,0.9)",
      )
  )
  fig.show()

Hierarchy: Station 8638610 is a Control Station.

Station is on the current NTDE (1983-2001)


In [181]:
# @title Final comparison between dpapi data and calculations in the notebook

trend_text = str(trend_text).replace('±', '+/-').split('+/-')[0].strip()
delta_por = round(delta_por, 3)
avg_last_5 = round(avg_last_5, 3)

# Organize the variables into a dictionary
table_data = {
    "Metric": [
        "Trend Rate",
        "POR Change",
        "NTDE Change"
    ],
    "Official Value": [
        trend_value,
        por_change,
        ntde_change
    ],
    "Notebook Value": [
        trend_text,
        delta_por,
        avg_last_5
    ]
}

# Create the pandas DataFrame
comparison_df = pd.DataFrame(table_data)

comparison_df

,Metric,Official Value,Notebook Value
0,Trend Rate,6.310,6.32
1,POR Change,0.502,0.502
2,NTDE Change,0.194,0.194


## **Takeaways**

###Thank you for taking the time to walk through this Sea Level Trend and Change Calculation Notebook

###Some key takeaways include:

*   How to access CO-OPS data from the CO-OPS API (here is a [URL builder](https://tidesandcurrents.noaa.gov/api-helper/url-generator.html) if you want to explore other CO-OPS data)
*   How to calculate trends using ARIMA modelling.
*   How CO-OPS calculates amount of change over the POR and since the NTDE